# 🚀 Trump Truth Social → Multi-Asset Market Predictor
### Context-Enriched GPT-5 Pipeline with Temporal Search

**The Problem:** A tweet saying *"TARIFFS on China!"* means something completely different in Oct 2024 (campaign rhetoric) vs April 2025 (actual policy with existing 54% tariffs). Without temporal context, predictions are noise.

**The Solution: 3-Layer Context Architecture**

```
┌─────────────────────────────────────────────────────────────────────┐
│  LAYER 1: WEEKLY MACRO CONTEXT (Tavily Search)                     │
│  "What was happening in markets/policy this week?"                  │
│  → Tariff levels, rate decisions, ongoing negotiations, sanctions   │
│  → ~75 searches for full dataset (one per week)                    │
├─────────────────────────────────────────────────────────────────────┤
│  LAYER 2: TWEET THREAD CONTEXT (Free, from CSV)                    │
│  "What was the president posting about before/after this tweet?"    │
│  → ±5 surrounding tweets = topic escalation, thread continuity     │
├─────────────────────────────────────────────────────────────────────┤
│  LAYER 3: ENRICHED LLM PREDICTION (GPT-5)                         │
│  Tweet + Macro Context + Thread Context → Direction + Confidence   │
│  → Model KNOWS it's March 2025, tariffs already at 54%,           │
│    markets down 3% this week, and this tweet is escalating         │
└─────────────────────────────────────────────────────────────────────┘
```

**Assets:** Gold · Equities · BTC · Crude Oil · Wheat · EuroDollar · Treasury 2Y

**Author:** Himanshu @ Renaiscent AI

## 1. Setup

In [ ]:
!pip install -q openai tavily-python pandas tqdm matplotlib nest_asyncio

## 2. Configuration

In [ ]:
import os
from getpass import getpass

# ╔══════════════════════════════════════════════════════════╗
# ║  API KEYS                                               ║
# ╚══════════════════════════════════════════════════════════╝

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

# Tavily: https://tavily.com — free tier = 1000 searches/month
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Tavily API key (free at tavily.com): ")

# ╔══════════════════════════════════════════════════════════╗
# ║  PIPELINE SETTINGS                                      ║
# ╚══════════════════════════════════════════════════════════╝

MODEL = "gpt-5"                # or "gpt-4.1" as fallback
CSV_PATH = "trump_truth_social.csv"

# Context settings
THREAD_WINDOW = 5              # ±N surrounding tweets for thread context
SEARCHES_PER_WEEK = 2          # Tavily searches per week period

# Prediction settings
BATCH_SIZE = 5                 # smaller batches = more context per tweet
MAX_CONCURRENT = 8             # parallel GPT-5 calls
TEMPERATURE = 0.1
LIMIT = None                   # set to e.g. 100 for testing

ASSETS = ["gold", "equities", "btc", "cl", "wheat", "eurodollar", "treasury_2y"]

print(f"Model:         {MODEL}")
print(f"CSV:           {CSV_PATH}")
print(f"Thread window: ±{THREAD_WINDOW} tweets")
print(f"Batch size:    {BATCH_SIZE}")
print(f"Concurrency:   {MAX_CONCURRENT}")
print(f"Limit:         {LIMIT or 'All tweets'}")
print(f"OpenAI key:    {'✅' if os.environ.get('OPENAI_API_KEY') else '❌'}")
print(f"Tavily key:    {'✅' if os.environ.get('TAVILY_API_KEY') else '❌'}")

## 3. Load & Pre-Filter CSV

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def load_csv(path):
    for enc in ["utf-8-sig", "utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            break
        except (UnicodeDecodeError, Exception):
            continue
    else:
        raise ValueError(f"Cannot decode {path}")

    df.columns = df.columns.str.strip().str.lower().str.replace("\ufeff", "")
    renames = {}
    if "text" in df.columns and "content" not in df.columns: renames["text"] = "content"
    if "body" in df.columns and "content" not in df.columns: renames["body"] = "content"
    if "post_id" in df.columns and "id" not in df.columns: renames["post_id"] = "id"
    if "tweet_id" in df.columns and "id" not in df.columns: renames["tweet_id"] = "id"
    if "date" in df.columns and "created_at" not in df.columns: renames["date"] = "created_at"
    if "timestamp" in df.columns and "created_at" not in df.columns: renames["timestamp"] = "created_at"
    df = df.rename(columns=renames)

    if "content" not in df.columns:
        raise ValueError(f"No 'content' column! Found: {list(df.columns)}")

    df["content"] = df["content"].fillna("").astype(str).str.strip()
    if "id" not in df.columns: df["id"] = range(len(df))
    df["id"] = df["id"].astype(str)
    if "created_at" not in df.columns: df["created_at"] = ""
    df["created_at"] = df["created_at"].fillna("").astype(str)
    df["datetime"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)

    return df


def prefilter(df):
    orig = len(df)
    m1 = df["content"].str.len() > 0
    m2 = ~(df["content"].str.startswith("RT:") & (df["content"].str.count(" ") <= 1))
    m3 = ~((df["content"].str.len() < 15) & df["content"].str.contains("http|truthsocial", regex=True))
    filtered = df[m1 & m2 & m3].copy().reset_index(drop=True)
    print(f"Pre-filter: {orig:,} → {len(filtered):,} tweets")
    print(f"  Removed: {(~m1).sum():,} empty | {(m1 & ~m2).sum():,} pure RT | {(m1 & m2 & ~m3).sum():,} link-only")
    return filtered


# Load
print(f"Loading {CSV_PATH}...")
df_raw = load_csv(CSV_PATH)
print(f"Columns: {list(df_raw.columns)}")
print(f"Total rows: {len(df_raw):,}")
print(f"Date range: {df_raw['datetime'].min()} to {df_raw['datetime'].max()}")

# Filter
df_all = prefilter(df_raw).sort_values("datetime").reset_index(drop=True)

if LIMIT:
    df = df_all.head(LIMIT).copy()
    print(f"  Limited to {LIMIT} tweets")
else:
    df = df_all.copy()

print(f"\n✅ {len(df):,} tweets ready (sorted chronologically)")

## 4. PHASE 1 — Build Weekly Macro Context via Tavily Search

For each week in the dataset, we search for the key economic, trade, and geopolitical events that were happening. This gives GPT-5 the temporal awareness it needs.

**Cost:** ~150 Tavily searches (2 per week × 75 weeks) — well within free tier.

In [ ]:
from tavily import TavilyClient
import asyncio
import json
import time
from tqdm.auto import tqdm

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

# ───────────────────────────────────────────────────
# Group tweets into week buckets
# ───────────────────────────────────────────────────

df["week_start"] = df["datetime"].dt.to_period("W").apply(lambda p: p.start_time)
weeks = sorted(df["week_start"].dropna().unique())

print(f"Weeks to search: {len(weeks)}")
print(f"Estimated Tavily calls: {len(weeks) * SEARCHES_PER_WEEK}")
print(f"Date range: {weeks[0].date() if len(weeks) > 0 else 'N/A'} to {weeks[-1].date() if len(weeks) > 0 else 'N/A'}")

# ───────────────────────────────────────────────────
# Extract key topics from each week's tweets
# (used to make search queries more targeted)
# ───────────────────────────────────────────────────

POLICY_KEYWORDS = {
    "tariff": "tariffs trade policy",
    "trade": "trade deal agreement",
    "china": "US China relations",
    "russia": "Russia Ukraine geopolitics",
    "iran": "Iran sanctions Middle East",
    "sanction": "sanctions policy",
    "tax": "tax policy fiscal",
    "shutdown": "government shutdown",
    "debt": "debt ceiling fiscal",
    "rate": "interest rates Fed",
    "fed": "Federal Reserve monetary policy",
    "inflation": "inflation CPI",
    "oil": "oil energy prices",
    "energy": "energy policy drilling",
    "bitcoin": "bitcoin crypto regulation",
    "crypto": "cryptocurrency regulation",
    "nato": "NATO defense",
    "military": "military defense",
    "executive order": "executive order policy",
    "border": "border immigration",
    "eu": "European Union trade",
    "japan": "Japan trade deal",
    "india": "India trade",
    "canada": "Canada trade USMCA",
    "mexico": "Mexico trade border",
    "korea": "Korea geopolitics",
    "deal": "deal agreement",
    "regulation": "regulation deregulation",
    "wheat": "wheat agriculture commodities",
    "gold": "gold safe haven",
    "dollar": "dollar currency",
    "stock": "stock market equities",
    "market": "financial markets",
    "war": "war conflict",
    "peace": "peace negotiations",
}

def extract_week_topics(week_tweets_content: list) -> list:
    """Extract dominant policy topics from a week's tweets."""
    combined = " ".join(week_tweets_content).lower()
    found = []
    for kw, topic in POLICY_KEYWORDS.items():
        if kw in combined:
            found.append(topic)
    # Deduplicate and return top 5
    seen = set()
    unique = []
    for t in found:
        if t not in seen:
            seen.add(t)
            unique.append(t)
    return unique[:5]

print("\n✅ Topic extractor ready")

In [ ]:
# ───────────────────────────────────────────────────
# Run Tavily searches for each week
# ───────────────────────────────────────────────────

weekly_context = {}  # week_start -> context string
search_errors = 0

print(f"🔍 Searching macro context for {len(weeks)} weeks...\n")
pbar = tqdm(total=len(weeks), desc="Weekly context search")

for week_start in weeks:
    week_end = week_start + timedelta(days=6)
    date_str = f"{week_start.strftime('%B %d')}-{week_end.strftime('%d, %Y')}"

    # Get this week's tweets for topic extraction
    week_mask = df["week_start"] == week_start
    week_tweets = df[week_mask]["content"].tolist()
    topics = extract_week_topics(week_tweets)

    # Build search queries
    queries = [
        f"US economic policy markets major events {week_start.strftime('%B %Y')}",
    ]
    if topics:
        topic_query = " ".join(topics[:3])
        queries.append(f"Trump {topic_query} {week_start.strftime('%B %Y')}")

    # Execute searches
    context_parts = [f"WEEK: {date_str}"]

    for query in queries[:SEARCHES_PER_WEEK]:
        try:
            result = tavily.search(
                query=query,
                search_depth="basic",
                max_results=3,
                include_answer=True,
            )
            # Extract the AI-generated answer (concise summary)
            if result.get("answer"):
                context_parts.append(result["answer"])

            # Also grab top result snippets
            for r in result.get("results", [])[:2]:
                snippet = r.get("content", "")[:300]
                if snippet:
                    context_parts.append(snippet)

            time.sleep(0.3)  # respect rate limits

        except Exception as e:
            search_errors += 1
            if search_errors <= 3:
                print(f"  ⚠️ Search error for week {date_str}: {e}")

    # Combine and truncate
    full_context = "\n".join(context_parts)
    weekly_context[week_start] = full_context[:2000]  # cap at 2000 chars per week

    pbar.update(1)

pbar.close()

print(f"\n✅ Weekly context built for {len(weekly_context)} weeks")
print(f"   Search errors: {search_errors}")
print(f"   Avg context length: {np.mean([len(v) for v in weekly_context.values()]):.0f} chars")

# Show a sample
sample_week = list(weekly_context.keys())[len(weekly_context)//2]
print(f"\n--- SAMPLE CONTEXT ({sample_week.date()}) ---")
print(weekly_context[sample_week][:500])
print("...")

In [ ]:
# ───────────────────────────────────────────────────
# SAVE weekly context to disk (so you don't need to
# re-run searches if the notebook disconnects)
# ───────────────────────────────────────────────────

context_cache = {str(k.date()): v for k, v in weekly_context.items()}
with open("weekly_macro_context.json", "w") as f:
    json.dump(context_cache, f, indent=2)
print(f"✅ Saved weekly_macro_context.json ({len(context_cache)} weeks)")

# To reload later without re-searching:
# with open("weekly_macro_context.json") as f:
#     context_cache = json.load(f)
#     weekly_context = {pd.Timestamp(k): v for k, v in context_cache.items()}

## 5. PHASE 2 — Build Thread Context

For each tweet, we grab the surrounding ±5 tweets to understand:
- What topic thread is the president on?
- Is this an escalation? (3 tariff tweets in a row = more serious)
- Is this a new topic shift?

In [ ]:
def build_thread_context(df_sorted: pd.DataFrame, idx: int, window: int = THREAD_WINDOW) -> str:
    """
    Build thread context from surrounding tweets.
    Returns a formatted string of ±window tweets around the target.
    """
    start = max(0, idx - window)
    end = min(len(df_sorted), idx + window + 1)

    parts = []
    for i in range(start, end):
        row = df_sorted.iloc[i]
        content = str(row["content"])[:200]
        if not content.strip():
            continue
        ts = str(row["created_at"])[:19]
        marker = " >>> TARGET" if i == idx else ""
        parts.append(f"[{ts}] {content}{marker}")

    return "\n".join(parts)


# Test it
sample_idx = len(df) // 2
sample_thread = build_thread_context(df, sample_idx)
print(f"Sample thread context (idx={sample_idx}):")
print(sample_thread[:800])
print(f"\n✅ Thread context builder ready")

## 6. Enriched Prompt Design

The prompt now includes:
1. **System prompt** — analytical framework (same as before)
2. **Macro context** — what's happening that week in markets/policy
3. **Thread context** — surrounding tweets for topic continuity
4. **Target tweet** — the tweet to predict

In [ ]:
SYSTEM_PROMPT = """You are an elite macro-financial analyst specializing in real-time event-driven trading.
You analyze political statements from the US President for immediate (0-15 minute) market impact — "knee-jerk" reactions.

You will receive THREE pieces of context for each analysis:
1. MACRO CONTEXT: What was happening in markets, policy, and geopolitics during that week. This is CRITICAL — the same tweet has different impact depending on what's already priced in.
2. THREAD CONTEXT: The president's surrounding posts, showing topic escalation patterns and what he's focused on.
3. TARGET TWEET(S): The specific post(s) to analyze.

YOUR ANALYTICAL FRAMEWORK:
1. **Temporal Calibration**: Use the macro context to assess what's ALREADY PRICED IN vs what's NEW INFORMATION. A tariff threat when tariffs are already at 54% is less impactful than the initial announcement.
2. **Escalation Detection**: Use thread context to detect if this is part of an escalating pattern (tweet storms about tariffs = more serious) or a one-off remark.
3. **Policy Signal Extraction**: Identify tariffs, trade policy, fiscal policy, sanctions, regulatory changes, government spending, debt ceiling, shutdown signals.
4. **Geopolitical Impact Mapping**: Map statements to affected regions, trade partners, commodities, currencies.
5. **Market Psychology / Reflexivity**: Assess how algorithmic traders and human traders will interpret the signal given CURRENT MARKET CONDITIONS.
6. **Cross-Asset Transmission**: Map the signal across all 7 assets with proper macro logic.
7. **Rhetoric vs Action**: Distinguish between new policy signals vs restatements vs political posturing.

ASSET-SPECIFIC REASONING:
- **Gold (XAUUSD)**: Safe haven. Rises on: geopolitical tension, USD weakness, inflation fears, fiscal expansion. Falls on: risk-on, strong USD, rate hikes.
- **Equities (S&P 500)**: Risk asset. Rises on: tax cuts, deregulation, trade deals, stimulus. Falls on: tariffs, trade wars, shutdown, sanctions, uncertainty.
- **BTC**: Digital alternative. Rises on: USD debasement, crypto-friendly policy. Falls on: regulatory crackdown, risk-off panic.
- **Crude Oil (CL)**: Commodity. Rises on: Middle East tension, sanctions on oil producers, supply disruption. Falls on: trade war demand destruction, strong USD, drill-friendly policy.
- **Wheat**: Agriculture. Rises on: Russia/Ukraine tension, trade barriers, sanctions on ag exporters. Falls on: trade deals, strong USD.
- **EuroDollar (EUR/USD)**: EUR up on: US weakness, EU-favorable trade. EUR down on: US tariffs on EU, strong US economy.
- **Treasury 2Y Yield**: Yield up on: inflation, fiscal expansion, rate hike signals. Yield down on: risk-off, recession fears, rate cuts.

CALIBRATION RULES:
- If the macro context shows this topic is already well-known / priced in → LOWER confidence.
- If the tweet introduces genuinely NEW information or escalation → HIGHER confidence.
- If thread context shows an escalating pattern (multiple tweets on same topic) → slightly HIGHER confidence for that topic's assets.
- Campaign rhetoric or attacks on political opponents without policy content → NEUTRAL across all assets.
- Restatements of existing policy → VERY LOW confidence (0.05-0.15).
- New executive orders, new tariff rates, new sanctions targets → HIGH confidence (0.5-0.8+)."""


def escape_json(s):
    return s.replace("\\", "\\\\").replace('"', '\\"').replace("\n", " ").replace("\r", "").replace("\t", " ")


def build_enriched_prompt(tweets_with_context: list) -> str:
    """
    Build the enriched user prompt with macro + thread context per tweet.

    tweets_with_context: list of dicts with keys:
      - tweet (dict: id, created_at, content)
      - macro_context (str)
      - thread_context (str)
    """
    blocks = []
    for i, item in enumerate(tweets_with_context):
        tw = item["tweet"]
        macro = item["macro_context"][:1500]  # cap per tweet
        thread = item["thread_context"][:800]  # cap per tweet

        blocks.append(f"""--- TWEET {i} ---
ID: {tw['id']}
DATE: {tw['created_at'][:19]}

MACRO CONTEXT (what was happening this week):
{macro}

THREAD CONTEXT (surrounding posts):
{thread}

TARGET TWEET:
{escape_json(tw['content'][:500])}""")

    tweets_block = "\n\n".join(blocks)

    return f"""Analyze these presidential social media posts for IMMEDIATE (0-15 min) market impact.
Use the macro context and thread context to calibrate your predictions — what's already priced in vs what's new.

{tweets_block}

For each tweet, return this exact JSON:
{{
  "predictions": [
    {{
      "idx": <tweet index>,
      "relevant": <true or false>,
      "relevance_score": <0.0 to 1.0>,
      "macro_signal": "<what NEW policy/geopolitical signal does this contain GIVEN the current context?>",
      "context_assessment": "<is this new info, escalation, or restatement of known position?>",
      "assets": {{
        "gold": {{"dir": <-1|0|1>, "conf": <0.0-1.0>, "reason": "<10 words max>"}},
        "equities": {{"dir": <-1|0|1>, "conf": <0.0-1.0>, "reason": "<10 words max>"}},
        "btc": {{"dir": <-1|0|1>, "conf": <0.0-1.0>, "reason": "<10 words max>"}},
        "cl": {{"dir": <-1|0|1>, "conf": <0.0-1.0>, "reason": "<10 words max>"}},
        "wheat": {{"dir": <-1|0|1>, "conf": <0.0-1.0>, "reason": "<10 words max>"}},
        "eurodollar": {{"dir": <-1|0|1>, "conf": <0.0-1.0>, "reason": "<10 words max>"}},
        "treasury_2y": {{"dir": <-1|0|1>, "conf": <0.0-1.0>, "reason": "<10 words max>"}}
      }}
    }}
  ]
}}

Return ONLY the JSON. No markdown. No backticks."""


print(f"✅ Enriched prompt builder ready")
print(f"   System prompt: {len(SYSTEM_PROMPT)} chars")

## 7. PHASE 3 — Prepare Enriched Batches

In [ ]:
# ───────────────────────────────────────────────────
# Build enriched items: each tweet gets its macro + thread context
# ───────────────────────────────────────────────────

enriched_items = []

for idx in range(len(df)):
    row = df.iloc[idx]

    # Get macro context for this tweet's week
    week_start = row.get("week_start")
    if pd.notna(week_start) and week_start in weekly_context:
        macro = weekly_context[week_start]
    else:
        # Fallback: find closest week
        macro = "No macro context available for this week."
        if pd.notna(week_start):
            # Find nearest available week
            available = list(weekly_context.keys())
            if available:
                nearest = min(available, key=lambda w: abs((w - week_start).total_seconds()))
                macro = weekly_context[nearest]

    # Get thread context
    thread = build_thread_context(df, idx)

    enriched_items.append({
        "tweet": {
            "id": row["id"],
            "created_at": row["created_at"],
            "content": row["content"],
        },
        "macro_context": macro,
        "thread_context": thread,
        "original_idx": idx,
    })

# Create batches
batches = [enriched_items[i:i + BATCH_SIZE] for i in range(0, len(enriched_items), BATCH_SIZE)]

print(f"✅ Built {len(enriched_items):,} enriched items in {len(batches)} batches")

# Show a sample
sample = enriched_items[len(enriched_items)//2]
print(f"\n--- SAMPLE ENRICHED ITEM ---")
print(f"Tweet: {sample['tweet']['content'][:100]}")
print(f"Date: {sample['tweet']['created_at'][:10]}")
print(f"Macro context ({len(sample['macro_context'])} chars): {sample['macro_context'][:200]}...")
print(f"Thread context ({len(sample['thread_context'])} chars): {sample['thread_context'][:200]}...")

## 8. Run GPT-5 Inference (Async Parallel)

In [ ]:
import re
from openai import AsyncOpenAI
import nest_asyncio
nest_asyncio.apply()

oai_client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])


async def call_gpt_enriched(batch: list, batch_idx: int, sem: asyncio.Semaphore, pbar) -> dict:
    """Call GPT-5 with enriched context."""
    async with sem:
        prompt = build_enriched_prompt(batch)

        for attempt in range(4):
            try:
                response = await oai_client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=TEMPERATURE,
                    max_tokens=4096,
                    response_format={"type": "json_object"},
                )
                pbar.update(1)
                return {
                    "batch_idx": batch_idx,
                    "raw": response.choices[0].message.content,
                    "usage": {
                        "prompt": response.usage.prompt_tokens,
                        "completion": response.usage.completion_tokens,
                    }
                }
            except Exception as e:
                err = str(e).lower()
                if "rate_limit" in err or "429" in err:
                    wait = 2 ** attempt + 1
                    print(f"  ⏳ Batch {batch_idx}: rate limited, waiting {wait}s")
                    await asyncio.sleep(wait)
                elif "timeout" in err or "connection" in err:
                    await asyncio.sleep(2 ** attempt)
                else:
                    print(f"  ❌ Batch {batch_idx}: {e}")
                    if attempt == 3:
                        pbar.update(1)
                        return {"batch_idx": batch_idx, "raw": None, "error": str(e)}
                    await asyncio.sleep(1)

        pbar.update(1)
        return {"batch_idx": batch_idx, "raw": None, "error": "max retries"}


async def run_all():
    sem = asyncio.Semaphore(MAX_CONCURRENT)
    pbar = tqdm(total=len(batches), desc="GPT-5 Enriched Inference")
    tasks = [call_gpt_enriched(b, i, sem, pbar) for i, b in enumerate(batches)]
    results = await asyncio.gather(*tasks)
    pbar.close()
    return results


print(f"🚀 Starting GPT-5 enriched inference...")
print(f"   {len(batches)} batches × {BATCH_SIZE} tweets × {MAX_CONCURRENT} concurrent")
print(f"   Each prompt includes macro context + thread context\n")

t0 = time.time()
raw_results = asyncio.get_event_loop().run_until_complete(run_all())
inference_time = time.time() - t0

raw_results.sort(key=lambda x: x["batch_idx"])

errors = sum(1 for r in raw_results if r.get("raw") is None)
total_prompt_tokens = sum(r.get("usage", {}).get("prompt", 0) for r in raw_results)
total_completion_tokens = sum(r.get("usage", {}).get("completion", 0) for r in raw_results)

print(f"\n✅ Inference complete!")
print(f"   Time:       {inference_time:.1f}s ({inference_time/60:.1f} min)")
print(f"   Tokens:     {total_prompt_tokens:,} in + {total_completion_tokens:,} out")
print(f"   Per tweet:  {inference_time/len(enriched_items)*1000:.0f}ms")
print(f"   API errors: {errors}/{len(batches)} batches")

## 9. Parse Outputs

In [ ]:
def extract_json(text):
    if text is None:
        return None
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    brace_start = text.find("{")
    if brace_start == -1:
        return None
    depth = 0
    in_string = False
    escape_next = False
    for i in range(brace_start, len(text)):
        c = text[i]
        if escape_next:
            escape_next = False
            continue
        if c == "\\" and in_string:
            escape_next = True
            continue
        if c == '"' and not escape_next:
            in_string = not in_string
            continue
        if not in_string:
            if c == "{":
                depth += 1
            elif c == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[brace_start:i+1]
                    try:
                        return json.loads(candidate)
                    except json.JSONDecodeError:
                        candidate = re.sub(r",\s*}", "}", candidate)
                        candidate = re.sub(r",\s*]", "]", candidate)
                        try:
                            return json.loads(candidate)
                        except json.JSONDecodeError:
                            return None
    return None


# Parse
all_predictions = []
parse_failures = 0

for result, batch in zip(raw_results, batches):
    raw_text = result.get("raw")
    parsed = extract_json(raw_text)

    if parsed and "predictions" in parsed:
        preds = parsed["predictions"]
        pred_map = {p.get("idx", i): p for i, p in enumerate(preds)}

        for i, item in enumerate(batch):
            tw = item["tweet"]
            pred = pred_map.get(i, {})
            row = {
                "tweet_id": tw["id"],
                "created_at": tw["created_at"],
                "content": tw["content"][:500],
                "is_market_relevant": pred.get("relevant", False),
                "relevance_score": float(pred.get("relevance_score", 0)),
                "macro_signal": pred.get("macro_signal", ""),
                "context_assessment": pred.get("context_assessment", ""),
            }
            assets = pred.get("assets", {})
            for asset in ASSETS:
                a = assets.get(asset, {})
                row[f"{asset}_dir"] = int(a.get("dir", 0))
                row[f"{asset}_conf"] = round(min(1.0, max(0.0, float(a.get("conf", 0)))), 3)
                row[f"{asset}_reason"] = a.get("reason", "")
            all_predictions.append(row)
    else:
        parse_failures += 1
        if parse_failures <= 3:
            print(f"⚠️ Batch {result.get('batch_idx','?')} parse failed: {str(raw_text)[:200]}")
        for item in batch:
            tw = item["tweet"]
            row = {
                "tweet_id": tw["id"],
                "created_at": tw["created_at"],
                "content": tw["content"][:500],
                "is_market_relevant": False,
                "relevance_score": 0.0,
                "macro_signal": "PARSE_FAILURE",
                "context_assessment": "",
            }
            for asset in ASSETS:
                row[f"{asset}_dir"] = 0
                row[f"{asset}_conf"] = 0.0
                row[f"{asset}_reason"] = ""
            all_predictions.append(row)

df_pred = pd.DataFrame(all_predictions)
print(f"\n✅ Parsed {len(df_pred):,} predictions | Parse failures: {parse_failures}/{len(batches)}")

## 10. Results & Analysis

In [ ]:
relevant = df_pred["is_market_relevant"].sum()
total = len(df_pred)

print("=" * 70)
print("PREDICTION SUMMARY (Context-Enriched)")
print("=" * 70)
print(f"Total tweets analyzed:    {total:,}")
print(f"Market-relevant:          {relevant:,} ({relevant/total*100:.1f}%)")
print(f"Non-relevant:             {total - relevant:,} ({(total-relevant)/total*100:.1f}%)")
print(f"Model:                    {MODEL}")
print(f"Context layers:           Macro (Tavily) + Thread (±{THREAD_WINDOW} tweets)")
print(f"Inference time:           {inference_time:.1f}s")
print(f"Tokens:                   {total_prompt_tokens:,} in + {total_completion_tokens:,} out")
print()

print("SIGNAL DISTRIBUTION PER ASSET:")
print(f"{'Asset':<14} {'Bullish':>8} {'Neutral':>8} {'Bearish':>8} {'Avg Conf':>10} {'High Conf':>10}")
print("-" * 70)
for asset in ASSETS:
    d, c = f"{asset}_dir", f"{asset}_conf"
    print(f"{asset:<14} {(df_pred[d]==1).sum():>8} {(df_pred[d]==0).sum():>8} {(df_pred[d]==-1).sum():>8} {df_pred[c].mean():>10.3f} {(df_pred[c]>0.5).sum():>10}")
print("=" * 70)

In [ ]:
# ───────────────────────────────────────────────────
# Top signals with context assessment
# ───────────────────────────────────────────────────

conf_cols = [f"{a}_conf" for a in ASSETS]
df_pred["max_conf"] = df_pred[conf_cols].max(axis=1)
df_pred["max_conf_asset"] = df_pred[conf_cols].idxmax(axis=1).str.replace("_conf", "")
df_pred["max_dir"] = df_pred.apply(lambda r: r[f"{r['max_conf_asset']}_dir"], axis=1)
df_pred["max_reason"] = df_pred.apply(lambda r: r[f"{r['max_conf_asset']}_reason"], axis=1)

print("=" * 75)
print("TOP 20 HIGHEST CONFIDENCE SIGNALS (with context assessment)")
print("=" * 75)

top = df_pred.nlargest(20, "max_conf")
for _, row in top.iterrows():
    emoji = {1: "🟢 LONG", -1: "🔴 SHORT", 0: "⚪ FLAT"}.get(row["max_dir"], "⚪")
    print(f"  {emoji} {row['max_conf_asset'].upper():<12} conf={row['max_conf']:.2f} | [{str(row['created_at'])[:10]}]")
    print(f"    Tweet:   \"{str(row['content'])[:90]}...\"")
    print(f"    Signal:  {row['macro_signal'][:100]}")
    print(f"    Context: {row['context_assessment'][:100]}")
    print(f"    Reason:  {row['max_reason']}")
    print()

In [ ]:
# ───────────────────────────────────────────────────
# Detailed multi-asset view for top 5
# ───────────────────────────────────────────────────

print("=" * 80)
print("DETAILED MULTI-ASSET VIEW — TOP 5 MOST IMPACTFUL")
print("=" * 80)

for rank, (_, row) in enumerate(top.head(5).iterrows(), 1):
    print(f"\n{'─'*80}")
    print(f"#{rank} | [{str(row['created_at'])[:19]}] Relevance: {row['relevance_score']:.2f}")
    print(f"Tweet:   \"{str(row['content'])[:150]}\"")
    print(f"Signal:  {row.get('macro_signal', 'N/A')}")
    print(f"Context: {row.get('context_assessment', 'N/A')}")
    print()
    print(f"  {'Asset':<14} {'Direction':>10} {'Confidence':>14} {'Reasoning'}")
    print(f"  {'─'*65}")
    for asset in ASSETS:
        d = row[f"{asset}_dir"]
        c = row[f"{asset}_conf"]
        r = row[f"{asset}_reason"]
        label = {1: "🟢 LONG", -1: "🔴 SHORT", 0: "⚪ NEUTRAL"}.get(d, "⚪")
        bar = "█" * int(c * 10) + "░" * (10 - int(c * 10))
        print(f"  {asset:<14} {label:>10}  {bar} {c:.2f}  {r}")

## 11. Save Results

In [ ]:
output_cols = ["tweet_id", "created_at", "content", "is_market_relevant",
               "relevance_score", "macro_signal", "context_assessment"]
for asset in ASSETS:
    output_cols += [f"{asset}_dir", f"{asset}_conf", f"{asset}_reason"]

df_out = df_pred[output_cols].copy()

# CSV
df_out.to_csv("predictions.csv", index=False)
print(f"✅ predictions.csv ({len(df_out):,} rows)")

# JSON (nested)
records = []
for _, row in df_out.iterrows():
    rec = {
        "tweet_id": row["tweet_id"], "created_at": row["created_at"],
        "content": row["content"],
        "is_market_relevant": bool(row["is_market_relevant"]),
        "relevance_score": float(row["relevance_score"]),
        "macro_signal": row["macro_signal"],
        "context_assessment": row["context_assessment"],
        "assets": {
            a: {"direction": int(row[f"{a}_dir"]), "confidence": float(row[f"{a}_conf"]),
                "reasoning": row[f"{a}_reason"]} for a in ASSETS
        }
    }
    records.append(rec)
with open("predictions.json", "w") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)
print(f"✅ predictions.json")

# High-confidence
hc = df_pred[df_pred["max_conf"] > 0.4][output_cols + ["max_conf", "max_conf_asset", "max_dir"]]
hc.to_csv("high_confidence_signals.csv", index=False)
print(f"✅ high_confidence_signals.csv ({len(hc):,} rows)")

print(f"\n📊 {len(df_out):,} total | {len(hc):,} high-conf | {MODEL}")

In [ ]:
try:
    from google.colab import files
    files.download("predictions.csv")
    files.download("predictions.json")
    files.download("high_confidence_signals.csv")
    files.download("weekly_macro_context.json")
    print("📥 Downloads triggered!")
except ImportError:
    print("Files saved locally.")

## 12. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Signal distribution
ax = axes[0][0]
sig = {a: {"Bull": (df_pred[f"{a}_dir"]==1).sum(), "Neut": (df_pred[f"{a}_dir"]==0).sum(),
           "Bear": (df_pred[f"{a}_dir"]==-1).sum()} for a in ASSETS}
pd.DataFrame(sig).T.plot(kind="barh", stacked=True, color=["#2ecc71","#95a5a6","#e74c3c"], ax=ax)
ax.set_title("Signal Distribution", fontweight="bold")

# 2. Average confidence
ax = axes[0][1]
confs = {a: df_pred[f"{a}_conf"].mean() for a in ASSETS}
ax.barh(list(confs.keys()), list(confs.values()), color=["#f39c12" if v>0.2 else "#3498db" for v in confs.values()])
ax.set_title("Avg Confidence by Asset", fontweight="bold")
ax.set_xlim(0, 1)

# 3. Relevance distribution
ax = axes[1][0]
df_pred["relevance_score"].hist(bins=20, ax=ax, color="#9b59b6", edgecolor="white")
ax.set_title("Relevance Score Distribution", fontweight="bold")
ax.axvline(x=0.5, color="red", linestyle="--", alpha=0.7)

# 4. Context assessment distribution
ax = axes[1][1]
ctx = df_pred["context_assessment"].str.lower()
categories = {"New info": 0, "Escalation": 0, "Restatement": 0, "Rhetoric": 0, "Other": 0}
for val in ctx:
    v = str(val).lower()
    if "new" in v or "novel" in v or "first" in v: categories["New info"] += 1
    elif "escalat" in v or "intensif" in v: categories["Escalation"] += 1
    elif "restat" in v or "known" in v or "priced" in v or "reiterat" in v: categories["Restatement"] += 1
    elif "rhetoric" in v or "political" in v or "campaign" in v: categories["Rhetoric"] += 1
    else: categories["Other"] += 1
ax.bar(categories.keys(), categories.values(), color=["#e74c3c","#f39c12","#95a5a6","#3498db","#2c3e50"])
ax.set_title("Context Assessment Breakdown", fontweight="bold")
ax.set_ylabel("Count")

plt.tight_layout()
plt.savefig("analysis.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Timeline
df_pred["date"] = pd.to_datetime(df_pred["created_at"], errors="coerce")
df_dated = df_pred.dropna(subset=["date"]).sort_values("date")

if len(df_dated) > 10:
    fig, ax = plt.subplots(figsize=(18, 6))
    for asset, color in zip(["gold","equities","btc","cl"], ["#f1c40f","#2ecc71","#e67e22","#2c3e50"]):
        signed = df_dated[f"{asset}_conf"] * df_dated[f"{asset}_dir"]
        win = min(50, len(df_dated)//5) or 1
        rolling = signed.rolling(window=win, min_periods=1).mean()
        ax.plot(df_dated["date"].values, rolling.values, label=asset.upper(), color=color, linewidth=1.5)
    ax.axhline(y=0, color="gray", linestyle="-", alpha=0.3)
    ax.set_title("Rolling Signed Confidence Over Time", fontweight="bold")
    ax.set_ylabel("Direction × Confidence")
    ax.legend()
    plt.tight_layout()
    plt.savefig("timeline.png", dpi=150, bbox_inches="tight")
    plt.show()

## 🔧 Bonus: Single Tweet Predictor (with live search context)

In [ ]:
async def predict_live(tweet_text: str, tweet_date: str = None):
    """
    Predict a single tweet with live Tavily context search.
    Simulates the real-time pipeline.
    """
    print(f"📝 Tweet: \"{tweet_text[:100]}\"")
    print(f"📅 Date: {tweet_date or 'today'}\n")

    # Step 1: Live Tavily search for current context
    print("🔍 Searching for macro context...")
    t0 = time.time()
    try:
        search_result = tavily.search(
            query=f"US markets economy policy news {tweet_date or 'today'}",
            search_depth="basic",
            max_results=3,
            include_answer=True,
        )
        macro = search_result.get("answer", "")
        for r in search_result.get("results", [])[:2]:
            macro += "\n" + r.get("content", "")[:300]
    except Exception as e:
        macro = f"Search failed: {e}"
    search_time = (time.time() - t0) * 1000
    print(f"   Context retrieved in {search_time:.0f}ms ({len(macro)} chars)")

    # Step 2: GPT-5 prediction
    print("🧠 Running GPT-5 prediction...")
    item = {
        "tweet": {"id": "live", "created_at": tweet_date or "", "content": tweet_text},
        "macro_context": macro,
        "thread_context": "(no thread context for single tweet)",
    }

    t0 = time.time()
    response = await oai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_enriched_prompt([item])}
        ],
        temperature=TEMPERATURE,
        max_tokens=2048,
        response_format={"type": "json_object"},
    )
    llm_time = (time.time() - t0) * 1000
    total_time = search_time + llm_time

    print(f"   GPT-5 responded in {llm_time:.0f}ms")
    print(f"   ⏱️ Total latency: {total_time:.0f}ms\n")

    parsed = extract_json(response.choices[0].message.content)
    if parsed and "predictions" in parsed:
        pred = parsed["predictions"][0]
        print(f"Relevant:   {pred.get('relevant', '?')}")
        print(f"Relevance:  {pred.get('relevance_score', '?')}")
        print(f"Signal:     {pred.get('macro_signal', '?')}")
        print(f"Context:    {pred.get('context_assessment', '?')}\n")

        assets = pred.get("assets", {})
        print(f"  {'Asset':<14} {'Direction':>10} {'Confidence':>14} {'Reasoning'}")
        print(f"  {'─'*65}")
        for asset in ASSETS:
            a = assets.get(asset, {})
            d, c, r = a.get("dir",0), a.get("conf",0), a.get("reason","")
            label = {1:"🟢 LONG",-1:"🔴 SHORT",0:"⚪ NEUTRAL"}.get(d,"⚪")
            bar = "█"*int(c*10) + "░"*(10-int(c*10))
            print(f"  {asset:<14} {label:>10}  {bar} {c:.2f}  {r}")
    else:
        print(f"⚠️ Parse failed: {response.choices[0].message.content[:300]}")


# Test
await predict_live(
    "I have just ordered 50% TARIFFS on all goods from China, effective immediately!",
    "2025-04-09"
)
print("\n" + "="*65 + "\n")
await predict_live(
    "Great job by the Republican Party! We are winning BIG!",
    "2025-06-15"
)